In [1]:
import pprint

import pandas as pd
import numpy as np
import geopandas as gpd
from shapely.geometry import Polygon

In [2]:

world_filepath = gpd.datasets.get_path('naturalearth_lowres')
world = gpd.read_file(world_filepath)

/tmp/ipykernel_1704551/947498767.py:1: FutureWarning: The geopandas.dataset module is deprecated and will be removed in GeoPandas 1.0. You can get the original 'naturalearth_lowres' data from https://www.naturalearthdata.com/downloads/110m-cultural-vectors/.
  world_filepath = gpd.datasets.get_path('naturalearth_lowres')


In [3]:
min_lon = -180
max_lon = 180
min_lat = -30
max_lat = 30
tropics = Polygon([(min_lon, min_lat), (min_lon, max_lat), (max_lon, max_lat, ), (max_lon, min_lat )])


In [4]:
tropical_countries = world.loc[world.intersects(tropics)]

In [5]:
#Equal-area projection
tropical_countries['area'] = \
    tropical_countries.to_crs('EPSG:6933').area / 1000000

/home/jan/miniconda3/envs/torch/lib/python3.11/site-packages/geopandas/geodataframe.py:1543: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [6]:
tropical_countries

,pop_est,continent,name,iso_a3,gdp_md_est,geometry,area
0,889953.0,Oceania,Fiji,FJI,5496,"MULTIPOLYGON (((180.00000 -16.06713, 180.00000...",1.928760e+04
1,58005463.0,Africa,Tanzania,TZA,63177,"POLYGON ((33.90371 -0.95000, 34.07262 -1.05982...",9.327793e+05
2,603253.0,Africa,W. Sahara,ESH,907,"POLYGON ((-8.66559 27.65643, -8.66512 27.58948...",9.666925e+04
4,328239523.0,North America,United States of America,USA,21433226,"MULTIPOLYGON (((-122.84000 49.00000, -120.0000...",9.509851e+06
7,8776109.0,Oceania,Papua New Guinea,PNG,24829,"MULTIPOLYGON (((141.00021 -2.60015, 142.73525 ...",4.645316e+05
...,...,...,...,...,...,...,...
167,5096159.0,Africa,Somaliland,SOL,17836,"POLYGON ((48.94820 11.41062, 48.94820 11.41062...",1.674732e+05
168,44269594.0,Africa,Uganda,UGA,35165,"POLYGON ((33.90371 -0.95000, 31.86617 -1.02736...",2.457603e+05
169,12626950.0,Africa,Rwanda,RWA,10354,"POLYGON ((30.41910 -1.13466, 30.81613 -1.69891...",2.336524e+04
175,1394973.0,North America,Trinidad and Tobago,TTO,24269,"POLYGON ((-61.68000 10.76000, -61.10500 10.890...",7.738923e+03


In [8]:
#df = pd.read_csv('../data/tmp/FAOSTAT_data_en_4-1-2024.csv')
df = pd.read_csv('../data/tmp/faostat_data_per_year.csv')

In [9]:
df.Item.unique()

array(['Cattle', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Oil palm fruit', 'Natural rubber in primary forms'], dtype=object)

In [10]:
#example
#Here the number is negative - 
#ie. area harvested is smaller in 2020 than in 2010; 
#therefore no expansion and i should enter 0 in the model
#cocoa_fiji = _df.loc[(_df.Element == 'Area harvested') & ( _df.Area == 'Fiji') & ( _df.Item == 'Cocoa beans')]
#harvested_2010 = cocoa_fiji.loc[cocoa_fiji.Year == 2010].Value.values[0] 
#harvested_2020 = cocoa_fiji.loc[cocoa_fiji.Year == 2020].Value.values[0] 


In [12]:
FAOSTAT_classes = ['Cattle', 'Oil palm fruit', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Natural rubber in primary forms']

In [13]:
country_name_mapping = {
    "Vietnam" : "Viet Nam",
    "Tanzania" : "United Republic of Tanzania",
    "Dem. Rep. Congo" : "Democratic Republic of the Congo",
    "Dominican Rep." : "Dominican Republic",
    "Bolivia" : "Bolivia (Plurinational State of)",
    "Venezuela" : "Venezuela (Bolivarian Republic of)",
    "Central African Rep.": "Central African Republic",
    "Eq. Guinea" : "Equatorial Guinea",
    "eSwatini" : "Eswatini",
    "Laos" : "Lao People's Democratic Republic",
    "Taiwan" : "China, Taiwan Province of",
    "Brunei" : "Brunei Darussalam",
    "S. Sudan" : "South Sudan",
    "Solomon Is." : "Solomon Islands"
}

In [14]:
names_to_edit = [key for key in country_name_mapping.keys()]
for name in names_to_edit:
    tropical_countries.loc[tropical_countries['name'] == name, 'name'] = country_name_mapping[name]
    

In [15]:
elements = {
    'Cattle' : {}, 
    'Oil palm fruit' : {}, 
    'Soya beans' : {}, 
    'Cocoa beans' : {}, 
    'Coffee, green' : {}, 
    'Natural rubber in primary forms' : {}
}

for idx, row in tropical_countries.iterrows():
    
    #try:
    #    country_name = country_name_mapping[row['name']]
    #except KeyError:
    country_name = row['name']
    
    _df = df.loc[df.Area == country_name]
    if len(_df) == 0:
        print('No FAOSTAT data found for ', row['name'])
    for _cls in FAOSTAT_classes:
        commodity_df = _df.loc[_df.Item == _cls]
        elements_available = commodity_df.Element.unique()
        
        for element in elements_available:
            try:
                elements[_cls][element] += 1
            except KeyError:
                elements[_cls][element] = 1

No FAOSTAT data found for  W. Sahara
No FAOSTAT data found for  Iran
No FAOSTAT data found for  Somaliland


In [16]:
pprint.pprint(elements)


{'Cattle': {'Stocks': 110},
 'Cocoa beans': {'Area harvested': 52, 'Production': 52, 'Yield': 50},
 'Coffee, green': {'Area harvested': 70, 'Production': 70, 'Yield': 69},
 'Natural rubber in primary forms': {'Area harvested': 35,
                                     'Production': 35,
                                     'Yield': 31},
 'Oil palm fruit': {'Area harvested': 43, 'Production': 43, 'Yield': 43},
 'Soya beans': {'Area harvested': 68, 'Production': 68, 'Yield': 62}}


In [32]:
df_filtered = df.loc[
    (df.Item == 'Cattle') |
    ((df.Item == 'Oil palm fruit') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Cocoa beans') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Coffee, green') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Natural rubber in primary forms') & (df.Element == 'Area harvested' )) |
    ((df.Item == 'Soya beans') & (df.Element == 'Area harvested' )) 
]

In [33]:
# 2.71 Mha - area under cultivation of cocoa for 2021 by Kalischek 

df_filtered.loc[
    ((df_filtered.Area == 'Ghana') & 
     (df_filtered.Year == 2020) & 
     (df_filtered.Item == "Cocoa beans")), 'Value'] = 2710000


In [34]:
df_filtered = df_filtered.loc[df_filtered.Year.isin([2000,2020])]

In [37]:
df_filtered

,Domain Code,Domain,Area Code (M49),Area,Element Code,Element,Item Code (CPC),Item,Year Code,Year,Unit,Value,Flag,Flag Description,Note
0,QCL,Crops and livestock products,4,Afghanistan,5111,Stocks,2111.0,Cattle,2000,2000,An,2900000.0,E,Estimated value,NaN
20,QCL,Crops and livestock products,4,Afghanistan,5111,Stocks,2111.0,Cattle,2020,2020,An,5326747.0,E,Estimated value,NaN
105,QCL,Crops and livestock products,12,Algeria,5111,Stocks,2111.0,Cattle,2000,2000,An,1595259.0,A,Official figure,NaN
125,QCL,Crops and livestock products,12,Algeria,5111,Stocks,2111.0,Cattle,2020,2020,An,1740183.0,A,Official figure,NaN
126,QCL,Crops and livestock products,24,Angola,5111,Stocks,2111.0,Cattle,2000,2000,An,4042000.0,X,Figure from international organizations,Unofficial figure
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24234,QCL,Crops and livestock products,716,Zimbabwe,5111,Stocks,2111.0,Cattle,2020,2020,An,5195154.0,A,Official figure,NaN
24235,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,1610.0,"Coffee, green",2000,2000,ha,6500.0,E,Estimated value,NaN
24295,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,1610.0,"Coffee, green",2020,2020,ha,2642.0,E,Estimated value,NaN
24298,QCL,Crops and livestock products,716,Zimbabwe,5312,Area harvested,141.0,Soya beans,2000,2000,ha,60650.0,A,Official figure,NaN


In [24]:
#TODO: 

## filter only years 2000 and 2020; 
## process as usual;
## write to file
## check that it works similar to the previous version (the only difference should be the palm oil)

##update the class names (palm oil) in dataset.py



In [444]:
#tropical_countries_names = [*[value for key, value in country_name_mapping.items()], *tropical_countries['name'].tolist()]


In [36]:
df_filtered = df_filtered.loc[df_filtered.Area.isin(tropical_countries['name'])]

In [38]:
df_filtered.to_csv('../data/tmp/faostat_reports_2000_2020_v2.csv', index=False)

In [40]:
def _get_commodity_expansion_rates(country, min_year, max_year=2020): #TODO: Max year should be 2022, this is temporary
        
    #TODO: this needs to be filtered so that for every commodity and year, there is only 1 element (either area, or production etc.)
    FAOSTAT_classes = ['Cattle', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Oil palm fruit', 'Natural rubber in primary forms']
    
    country_data = df_filtered.loc[df_filtered.Area == country]

    values = []

    for _class in FAOSTAT_classes:
        try:
            production_at_the_start = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(min_year)), 'Value' ].item()
            production_at_the_end = country_data.loc[(country_data.Item == _class) & (country_data.Year == int(max_year)), 'Value' ].item()
            expansion = production_at_the_end - production_at_the_start
        except ValueError: #Signifies that one (or both) of the values are missing
            expansion = 0
        values.append(expansion)

    return values

In [41]:
vals = {}

for country in tropical_countries.name.unique():
    vals[country] = _get_commodity_expansion_rates(country, min_year=2000)

In [48]:

FAOSTAT_classes = ['Cattle', 'Soya beans', 'Cocoa beans', 'Coffee, green',
       'Oil palm fruit', 'Natural rubber in primary forms']

In [54]:
tmp_df = pd.DataFrame(vals).T
tmp_df.columns = FAOSTAT_classes

In [55]:
tmp_df

,Cattle,Soya beans,Cocoa beans,"Coffee, green",Oil palm fruit,Natural rubber in primary forms
Fiji,-241699.0,0.0,-639.0,-8.0,0.0,0.0
United Republic of Tanzania,13551841.0,36162.0,16484.0,116960.0,958.0,0.0
W. Sahara,0.0,0.0,0.0,0.0,0.0,0.0
United States of America,-4405700.0,4125820.0,0.0,0.0,0.0,0.0
Papua New Guinea,4668.0,0.0,-2474.0,-43357.0,148624.0,662.0
...,...,...,...,...,...,...
Somaliland,0.0,0.0,0.0,0.0,0.0,0.0
Uganda,8549833.0,-54000.0,55465.0,349885.0,0.0,0.0
Rwanda,876376.0,18945.0,0.0,-5147.0,0.0,0.0
Trinidad and Tobago,5575.0,0.0,-11278.0,765.0,0.0,0.0


In [56]:
areas = []
for idx, row in tmp_df.iterrows():
    _name = row.name
    area = tropical_countries.loc[tropical_countries['name'] == _name]['area'].item()
    areas.append(area)

tmp_df['country_area'] = areas

In [57]:
tmp_df
tmp_df.loc[tmp_df.index == "Viet Nam"]



,Cattle,Soya beans,Cocoa beans,"Coffee, green",Oil palm fruit,Natural rubber in primary forms,country_area
Viet Nam,2102647.0,-82536.0,0.0,160663.0,0.0,497264.0,336019.374187


In [58]:
def normalize_by_col(df, cols_to_normalize, norm_column):
    """For each column in df from cols_to_normalize, divide by the value in norm_column. Return normalized df"""
    df_normalized = df.copy()
    for col in cols_to_normalize:
        df_normalized[col] = df_normalized[col] / df_normalized[norm_column]
    return df_normalized

In [66]:
norm_df = normalize_by_col(tmp_df, FAOSTAT_classes, 'country_area')

In [67]:
norm_df

,Cattle,Soya beans,Cocoa beans,"Coffee, green",Oil palm fruit,Natural rubber in primary forms,country_area
Fiji,-12.531313,0.000000,-0.033130,-0.000415,0.000000,0.000000,1.928760e+04
United Republic of Tanzania,14.528453,0.038768,0.017672,0.125389,0.001027,0.000000,9.327793e+05
W. Sahara,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,9.666925e+04
United States of America,-0.463278,0.433847,0.000000,0.000000,0.000000,0.000000,9.509851e+06
Papua New Guinea,0.010049,0.000000,-0.005326,-0.093335,0.319944,0.001425,4.645316e+05
...,...,...,...,...,...,...,...
Somaliland,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.674732e+05
Uganda,34.789324,-0.219726,0.225687,1.423684,0.000000,0.000000,2.457603e+05
Rwanda,37.507676,0.810820,0.000000,-0.220284,0.000000,0.000000,2.336524e+04
Trinidad and Tobago,0.720384,0.000000,-1.457309,0.098851,0.000000,0.000000,7.738923e+03


In [68]:
pd.set_option('display.float_format', lambda x: '%.3f' % x)

### Normalization v2

just divide by max value for each of palm oil, cattle, area harvested

In [70]:
max_vals = norm_df.max()

In [71]:
norm_df['Cattle'] = norm_df['Cattle']/max_vals['Cattle']


max_area = max_vals[['Soya beans', 'Cocoa beans', 'Coffee, green', 'Oil palm fruit',
       'Natural rubber in primary forms']].max()

for _cls in ['Soya beans', 'Cocoa beans', 'Coffee, green', 'Oil palm fruit',
       'Natural rubber in primary forms']:
    
    norm_df[_cls] = norm_df[_cls]/max_area


In [72]:
norm_df['country'] = norm_df.index

In [73]:
norm_df.to_csv('../data/tmp/faostat_reports_v2.csv', index=False)

## Read and explore the normalized data

In [74]:
norm_df = pd.read_csv('../data/tmp/faostat_reports.csv')
norm_df_v2 = pd.read_csv('../data/tmp/faostat_reports_v2.csv')

In [82]:
#norm_df['geometry'] = gpd.GeoSeries.from_wkt(norm_df['geometry'])

#gdf = gpd.GeoDataFrame(norm_df)

In [75]:
tmp = norm_df.set_index('country')
tmp2 = norm_df_v2.set_index('country')

In [76]:
tmp

,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms,country_area
country,,,,,,,
Fiji,-0.334,0.000,0.000,-0.005,-0.000,0.000,19287.604
United Republic of Tanzania,0.387,-0.000,0.006,0.003,0.018,0.000,932779.346
W. Sahara,0.000,0.000,0.000,0.000,0.000,0.000,96669.250
United States of America,-0.012,0.000,0.062,0.000,0.000,0.000,9509850.934
Papua New Guinea,0.000,0.030,0.000,-0.001,-0.013,0.000,464531.614
...,...,...,...,...,...,...,...
Somaliland,0.000,0.000,0.000,0.000,0.000,0.000,167473.218
Uganda,0.928,0.000,-0.032,0.033,0.205,0.000,245760.253
Rwanda,1.000,0.000,0.117,0.000,-0.032,0.000,23365.244


In [80]:
tmp2

,Cattle,Soya beans,Cocoa beans,"Coffee, green",Oil palm fruit,Natural rubber in primary forms,country_area
country,,,,,,,
Fiji,-0.334,0.000,-0.005,-0.000,0.000,0.000,19287.604
United Republic of Tanzania,0.387,0.006,0.003,0.018,0.000,0.000,932779.346
W. Sahara,0.000,0.000,0.000,0.000,0.000,0.000,96669.250
United States of America,-0.012,0.062,0.000,0.000,0.000,0.000,9509850.934
Papua New Guinea,0.000,0.000,-0.001,-0.013,0.046,0.000,464531.614
...,...,...,...,...,...,...,...
Somaliland,0.000,0.000,0.000,0.000,0.000,0.000,167473.218
Uganda,0.928,-0.032,0.033,0.205,0.000,0.000,245760.253
Rwanda,1.000,0.117,0.000,-0.032,0.000,0.000,23365.244


In [24]:
norm_df.to_csv('../data/tmp/faostat_reports.csv', index=False)

In [22]:
norm_df['country'] = tropical_countries['name'].reset_index(drop=True)

In [9]:
norm_df.loc[norm_df['country'] == "Malaysia"]



,Cattle,Palm oil,Soya beans,Cocoa beans,"Coffee, green",Natural rubber in primary forms,country_area,country
97,-0.003103,1.0,0.0,-0.029833,-0.011306,-0.124159,338220.757641,Malaysia


In [103]:
df_filtered['Flag Description'].unique()

array(['Official figure', 'Estimated value',
       'Missing value (data cannot exist, not applicable)',
       'Imputed value', 'Figure from international organizations'],
      dtype=object)

In [74]:
df = pd.read_csv("/home/jan/Documents/EPFL/project_2/data/labels/dataset_v4/labels_55k.csv")
df['geometry'] = gpd.GeoSeries.from_wkt(df['geometry'])

gdf = gpd.GeoDataFrame(df)

In [78]:
names = gdf.country.unique()

In [79]:
_dict = {}
test_countries = ['Ecuador', 'Malaysia', 'Thailand']
for name in names:
    if name not in test_countries:
        _dict[name] = "train"
    else:
        _dict[name] = "val"

In [80]:
import pprint
pprint.pprint(_dict)

{'Angola': 'train',
 'Australia': 'train',
 'Bahamas': 'train',
 'Bangladesh': 'train',
 'Belize': 'train',
 'Benin': 'train',
 'Bhutan': 'train',
 'Bolivia (Plurinational State of)': 'train',
 'Brazil': 'train',
 'Brunei Darussalam': 'train',
 'Burundi': 'train',
 'Cambodia': 'train',
 'Cameroon': 'train',
 'Central African Republic': 'train',
 'China': 'train',
 'Colombia': 'train',
 'Congo': 'train',
 'Costa Rica': 'train',
 'Cuba': 'train',
 "Côte d'Ivoire": 'train',
 'Democratic Republic of the Congo': 'train',
 'Dominican Republic': 'train',
 'Ecuador': 'val',
 'El Salvador': 'train',
 'Equatorial Guinea': 'train',
 'Ethiopia': 'train',
 'Fiji': 'train',
 'France': 'train',
 'Gabon': 'train',
 'Ghana': 'train',
 'Guatemala': 'train',
 'Guinea': 'train',
 'Guyana': 'train',
 'Haiti': 'train',
 'Honduras': 'train',
 'India': 'train',
 'Indonesia': 'train',
 'Jamaica': 'train',
 'Kenya': 'train',
 "Lao People's Democratic Republic": 'train',
 'Liberia': 'train',
 'Madagascar': 'trai